# PotholeMeasure — end-to-end demo

Walks through a single-image inference and visualises each stage of the pipeline:

1. Instance segmentation (YOLOv8-seg).
2. Metric depth map (Depth Anything V2 Metric Outdoor).
3. RANSAC road plane on non-pothole 3D points.
4. Per-mask depth = p95 of `|signed distance to plane|`.
5. Per-mask area in m² via the ground-plane homography.
6. Severity bucket per GOST R 50597-2017.

**Before running:** drop a trained checkpoint into `experiments/checkpoints/best.pt` and update
`configs/default.yaml → paths.segmentation_weights` if needed. A sample image path can be passed via
`IMAGE_PATH` below.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.pipeline import PotholePipeline
from src.visualize import draw_results, plot_depth_comparison

In [ ]:
CFG_PATH = ROOT / "configs/default.yaml"
IMAGE_PATH = ROOT / "data/raw/sample.jpg"  # replace with a real image

pipeline = PotholePipeline.from_config(CFG_PATH)

image = cv2.imread(str(IMAGE_PATH))
assert image is not None, f"could not read {IMAGE_PATH}"
frame = pipeline.process(image, image_path=str(IMAGE_PATH))
print(frame.to_json_dict())

In [ ]:
overlay = draw_results(image, frame.potholes)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Severity overlay")
plt.show()

In [ ]:
depth_map = pipeline.depth_estimator.predict(image)
mask_union = np.zeros(depth_map.shape, dtype=bool)
for p in frame.potholes:
    if p.mask is not None:
        mask_union |= p.mask

plot_depth_comparison(image, depth_map, mask_union, plane=frame.plane)

### Ablation table for the paper

```bash
python scripts/evaluate.py \
    --config configs/default.yaml \
    --gt data/annotations/test_gt.json \
    --output experiments/results/ablation.csv --latex
```

Produces `experiments/results/ablation.csv` (and `.tex`) with MAE / RMSE / severity F1
for the three recipes: `midas_relative_scaled`, `metric_no_plane`, `metric_plane_offset` (ours).